# LSTM time step regression framing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pandas import read_csv
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# convert an array of values into a dataset matrix
def create_dataset(dataset, look_back=1):
	dataX, dataY = [], []
	for i in range(len(dataset)-look_back-1):
		a = dataset[i:(i+look_back), 0]
		dataX.append(a)
		dataY.append(dataset[i + look_back, 0])
	return np.array(dataX), np.array(dataY)

# fix random seed for reproducibility
tf.random.set_seed(42)

# load the dataset
dataframe = read_csv('airline-passengers.csv', usecols=[1], engine='python')
dataset = dataframe.values
dataset = dataset.astype('float32')

In [ ]:
plt.plot(dataset)
plt.show()

In [ ]:
# normalize the dataset
scaler = MinMaxScaler(feature_range=(0, 1))
dataset = scaler.fit_transform(dataset)

In [ ]:
# split into train and test sets
train_size = int(len(dataset) * 0.7)
test_size = len(dataset) - train_size
train, test = dataset[0:train_size,:], dataset[train_size:len(dataset),:]

# reshape into X=t and Y=t+1
look_back = 12
trainX, trainY = create_dataset(train, look_back)
testX, testY = create_dataset(test, look_back)

# reshape input to be [samples, time steps, features] - LSTM expects 3D input
trainX = np.reshape(trainX, (trainX.shape[0], trainX.shape[1], 1))
testX = np.reshape(testX, (testX.shape[0], testX.shape[1], 1))

In [ ]:
# create and fit the LSTM network
model = Sequential()
model.add(LSTM(32, 
               input_shape=(look_back, 1)))
model.add(Dense(1))
model.compile(loss='mean_squared_error', 
              optimizer='adam')
model.fit(trainX, 
          trainY, 
          epochs=100, 
          batch_size=16, 
          verbose=2)

In [ ]:
# You can add another layer

# model = Sequential()
# model.add(LSTM(32, return_sequences=True, input_shape=(1, look_back)))
# model.add(LSTM(16))
# model.add(Dense(1))

In [ ]:
# make predictions
trainPredict = model.predict(trainX)
testPredict = model.predict(testX)

In [ ]:
# invert predictions
trainPredict = scaler.inverse_transform(trainPredict)
trainY = scaler.inverse_transform([trainY])
testPredict = scaler.inverse_transform(testPredict)
testY = scaler.inverse_transform([testY])

In [ ]:
# calculate root mean squared error
trainScore = np.sqrt(mean_squared_error(trainY[0], trainPredict[:,0]))
print('Train Score: %.2f RMSE' % (trainScore))
testScore = np.sqrt(mean_squared_error(testY[0], testPredict[:,0]))
print('Test Score: %.2f RMSE' % (testScore))

In [ ]:
# shift train predictions for plotting
trainPredictPlot = np.empty_like(dataset)
trainPredictPlot[:, :] = np.nan
trainPredictPlot[look_back:len(trainPredict)+look_back, :] = trainPredict

In [ ]:
# shift test predictions for plotting
testPredictPlot = np.empty_like(dataset)
testPredictPlot[:, :] = np.nan
testPredictPlot[len(trainPredict)+(look_back*2)+1:len(dataset)-1, :] = testPredict

In [ ]:
# plot baseline and predictions
plt.plot(scaler.inverse_transform(dataset))
plt.plot(trainPredictPlot)
plt.plot(testPredictPlot)
plt.show()

## USING NEURAL NETWORK

In [ ]:
# Split
train_size = int(len(dataset) * 0.7)
train, test = dataset[:train_size, :], dataset[train_size:, :]

# Supervised framing
look_back = 12   # use 1 year of history
train_X, train_Y = create_dataset(train, look_back)
test_X, test_Y = create_dataset(test, look_back)

Define the NN model

In [ ]:
model = Sequential()
model.add(Dense(64, activation="relu", input_shape=(look_back,)))
model.add(Dense(32, activation="relu"))
model.add(Dense(1))

model.compile(loss="mse", optimizer="adam")

In [ ]:
# Train
model.fit(train_X, train_Y, epochs=200, batch_size=16, verbose=2)

In [ ]:
# Predict
train_Predict = model.predict(train_X)
test_Predict = model.predict(test_X)

In [ ]:
# Invert scaling

train_Predict = scaler.inverse_transform(train_Predict)
train_Y_inv = scaler.inverse_transform(train_Y.reshape(-1, 1))

test_Predict = scaler.inverse_transform(test_Predict)
test_Y_inv = scaler.inverse_transform(test_Y.reshape(-1, 1))

In [ ]:
# Metrics

trainRMSE = np.sqrt(mean_squared_error(train_Y_inv[:, 0], train_Predict[:, 0]))
testRMSE = np.sqrt(mean_squared_error(test_Y_inv[:, 0], test_Predict[:, 0]))

trainMAE = mean_absolute_error(train_Y_inv[:, 0], train_Predict[:, 0])
testMAE = mean_absolute_error(test_Y_inv[:, 0], test_Predict[:, 0])

r2 = r2_score(test_Y_inv[:, 0], test_Predict[:, 0])

print(f"Train RMSE: {trainRMSE:.2f}")
print(f"Test RMSE : {testRMSE:.2f}")
print(f"Test MAE  : {testMAE:.2f}")
print(f"Test R2   : {r2:.3f}")

In [ ]:
# Plotting
trainPredictPlot = np.empty_like(dataset)
trainPredictPlot[:, :] = np.nan
trainPredictPlot[look_back:len(train_Predict) + look_back, :] = train_Predict

testPredictPlot = np.empty_like(dataset)
testPredictPlot[:, :] = np.nan
testPredictPlot[len(train_Predict) + (look_back * 2) + 1:len(dataset) - 1, :] = test_Predict

plt.figure(figsize=(10,6))
plt.plot(scaler.inverse_transform(dataset), label="Actual")
plt.plot(trainPredictPlot, linestyle="--", label="Train Predict")
plt.plot(testPredictPlot, linestyle="--", label="Test Predict")
plt.legend()
plt.title("NN - Time Series Forecast")
plt.show()